We synthesize a pruning rule over instance symbols, so that whenever the rule fires, pruning is sound for the swap comparison. In order to do so, we supply a Context-Free Grammar $G = (N, \Sigma, P, S)$ defined over a set of nonterminals $N$, terminals (symbols) $\Sigma$, productions $P$ and start-node $S$.

Here, we run with nonterminals for $\texttt{Rule}$ and $\texttt{Atom}$. Rules consist of conjunctions over comparisons, where comparisons compare classwise-pairs of attributes for $i$ and $j$ respectively.

$
N = \{\texttt{Rule},\text{Atom}\}, \quad S = \texttt{Rule}
$

Explicitly, the productions of the CFG are as follows:

$
\texttt{Rule} \mapsto \texttt{Atom} \mid (\texttt{Atom} \land \texttt{Atom}) \mid (\texttt{Atom} \land \texttt{Atom} \land \texttt{Atom})
$

$
\texttt{Atom} \mapsto (x \le y) \mid (x = y)
$

where typed comparable pairs $(x,y)$ from release/window/cost symbols and inbound/outbound separations.

The synthesized object is a Boolean conjunction schema over these atoms, instantiated per problem instance. First, we synthesise such a rule for complete orders over makespan.


In [16]:
from synthesis.grammar import *
from synthesis.synth import *
from synthesis import *

problem = make_rsp_swap_problem(timeout_ms = 10000, objective_name = 'makespan')

log(f"Symbol Set: [{', '.join(symbol.name.replace('_', '') for symbol in problem.symbols)}]")

# Construct the grammar.
conj = NonTerminal("Rule", sort = problem.env.bool_sort)
cmp = NonTerminal("Atom", sort = problem.env.bool_sort)

by_name = {symbol.name: symbol for symbol in problem.symbols}

flat = lambda xs: [y for x in xs for y in (flat(x) if isinstance(x, list) else [x])]
names = set(by_name)

prefixes = sorted({name[:-2] for name in names if name.endswith("_i") and not name.startswith("D_") and name[:-2] != "T"})
comparable_pairs = [
    *[(f"{p}_i", f"{p}_j") for p in prefixes if f"{p}_j" in names],
    *[pair for pair in (("D_i_x", "D_j_x"), ("D_x_i", "D_x_j")) if pair[0] in names and pair[1] in names],
]

grammar = Grammar(
    nonterminals=(conj, cmp),
    terminals=problem.symbols,
    start=conj,
    productions=(
        conj >> (cmp | (cmp & cmp) | (cmp & cmp & cmp)),
        cmp >> Choice(tuple(flat(
            [[by_name[l] <= by_name[r], by_name[r] <= by_name[l], by_name[l].eq(by_name[r])] for l, r in comparable_pairs]
        ))),
    ),
)

# Binds Terminal symbols to problem variables.
set_smt_env(
    symbol_table={symbol.name: symbol.formal for symbol in problem.symbols},
)
grammar = grammar.to_cvc5(problem.env.solver)


# Run Synthesis
result = synthesize_pruning_rule(
    problem,
    grammar=grammar,
    require_nonvacuous=True,
)

if result.rule_solution is not None:
    print(f"\n{result.rule_solution}")

[14:03:53] Symbol Set: [Ri, Rj, LTi, LTj, LCi, LCj, ETi, ETj, ECi, ECj, Bi, Bj, Ci, Cj, Dix, Djx, Dxi, Dxj]
[14:03:53] Invoking Synthesis (This may take some time)
[14:03:55] Synthesis Complete, Result: (SOLUTION)

(
  (define-fun prune ((R_i Int) (R_j Int) (LT_i Int) (LT_j Int) (LC_i Int) (LC_j Int) (ET_i Int) (ET_j Int) (EC_i Int) (EC_j Int) (B_i Int) (B_j Int) (C_i Int) (C_j Int) (D_i_x Int) (D_j_x Int) (D_x_i Int) (D_x_j Int)) Bool (and (= D_i_x D_j_x) (<= R_i R_j) (= D_x_i D_x_j)))
)


Now, we use the same formulation to synthesise a complete order for the delay objective.

In [15]:
from synthesis.grammar import *
from synthesis.synth import *
from synthesis import *

problem = make_rsp_swap_problem(timeout_ms = 10000, objective_name = 'delay')

log(f"Symbol Set: [{', '.join(symbol.name.replace('_', '') for symbol in problem.symbols)}]")

# Construct the grammar.
conj = NonTerminal("Rule", sort = problem.env.bool_sort)
cmp = NonTerminal("Atom", sort = problem.env.bool_sort)

by_name = {symbol.name: symbol for symbol in problem.symbols}

flat = lambda xs: [y for x in xs for y in (flat(x) if isinstance(x, list) else [x])]
names = set(by_name)

prefixes = sorted({name[:-2] for name in names if name.endswith("_i") and not name.startswith("D_") and name[:-2] != "T"})
comparable_pairs = [
    *[(f"{p}_i", f"{p}_j") for p in prefixes if f"{p}_j" in names],
    *[pair for pair in (("D_i_x", "D_j_x"), ("D_x_i", "D_x_j")) if pair[0] in names and pair[1] in names],
]

grammar = Grammar(
    nonterminals=(conj, cmp),
    terminals=problem.symbols,
    start=conj,
    productions=(
        conj >> (cmp | (cmp & cmp) | (cmp & cmp & cmp)),
        cmp >> Choice(tuple(flat(
            [[by_name[l] <= by_name[r], by_name[r] <= by_name[l], by_name[l].eq(by_name[r])] for l, r in comparable_pairs]
        ))),
    ),
)

# Binds Terminal symbols to problem variables.
set_smt_env(
    symbol_table={symbol.name: symbol.formal for symbol in problem.symbols},
)
grammar = grammar.to_cvc5(problem.env.solver)


# Run Synthesis
result = synthesize_pruning_rule(
    problem,
    grammar=grammar,
    require_nonvacuous=True,
)

if result.rule_solution is not None:
    print(f"\n{result.rule_solution}")

[23:06:23] Symbol Set: [Ri, Rj, LTi, LTj, LCi, LCj, ETi, ETj, ECi, ECj, Bi, Bj, Ci, Cj, Dix, Djx, Dxi, Dxj]
[23:06:23] Invoking Synthesis (This may take some time)
[23:06:26] Synthesis Complete, Result: (SOLUTION)

(
  (define-fun prune ((R_i Int) (R_j Int) (LT_i Int) (LT_j Int) (LC_i Int) (LC_j Int) (ET_i Int) (ET_j Int) (EC_i Int) (EC_j Int) (B_i Int) (B_j Int) (C_i Int) (C_j Int) (D_i_x Int) (D_j_x Int) (D_x_i Int) (D_x_j Int)) Bool (and (= R_i R_j) (= D_i_x D_j_x) (= D_x_i D_x_j)))
)
